# 09 — Extensión: ¿la minuta predice la *próxima* decisión? (target N+1)

Los experimentos 1–4b modelan la relación **concurrente** `f(minuta_N) → decisión_N`: clasifican la
decisión tomada en *esa misma* reunión. Como la minuta se publica ~3 semanas después, es un estudio
explicativo, no una predicción a futuro.

Este notebook testea una hipótesis más ambiciosa y genuinamente predictiva: **¿el tono/contenido de
la minuta N anticipa la decisión de la reunión siguiente (N+1)?** (forward guidance / momentum). Para
eso re-etiquetamos `minuta_N → decisión_{N+1}` y re-corremos la escalera completa
(TF-IDF → Word2Vec → FinBERT FT).

**El rival a vencer no es "siempre hold" sino la persistencia** ("la Fed repite la próxima vez lo que
hizo esta vez"). Si el texto supera la persistencia, la minuta tiene señal forward-looking real; si
no, las decisiones N+1 son dominadas por la inercia.

**Aislado del resto:** re-etiqueta en memoria desde `data/processed/fomc_dataset.csv`; no toca el CSV
ni los notebooks 02–08 (Exp 1–4b ya entregados).

**Entornos** (igual que el proyecto): relabel + baselines + TF-IDF en cualquier env sklearn; Word2Vec
en el venv py3.12 (gensim + GoogleNews); FinBERT FT en el venv CUDA. Cada sección graba su resultado
en `reports/n1_results.json`, que la celda final consolida (así no importa que cada sección corra en
un kernel distinto).

## 0. Setup y helpers compartidos

Importes base (sklearn), rutas, `LABEL_ORDER`, el helper `evaluar` (idéntico a 04–07) y un acumulador
de resultados en disco (`record_result`) para que las tres secciones —que corren en entornos
distintos— junten sus números sin transcribir a mano.

In [ ]:
import json
import numpy as np
import pandas as pd
from pathlib import Path

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.dummy import DummyClassifier
from sklearn.metrics import (classification_report, confusion_matrix,
                             f1_score, accuracy_score, recall_score)

sns.set_theme(style='whitegrid', palette='Set2')
pd.options.display.max_colwidth = 200

BASE_DIR       = Path('..').resolve()
DATA_PROCESSED = BASE_DIR / 'data' / 'processed'
MINUTES_DIR    = BASE_DIR / 'data' / 'raw' / 'minutes'
REPORTS_DIR    = BASE_DIR / 'reports' / 'figures'
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

LABEL_ORDER = ['hike', 'cut', 'hold']

def save_fig(name):
    plt.savefig(REPORTS_DIR / name, dpi=120, bbox_inches='tight')

# Helper de evaluación (idéntico a notebooks 04-07).
def evaluar(y_true, y_pred, titulo, fname=None):
    clases = [c for c in LABEL_ORDER if c in set(y_true)]
    f1 = f1_score(y_true, y_pred, labels=clases, average='macro')
    print(titulo)
    print('accuracy :', round(accuracy_score(y_true, y_pred), 4))
    print(f'f1_macro : {round(f1, 4)}  (sobre {len(clases)} clases: {clases})')
    print(classification_report(y_true, y_pred, labels=LABEL_ORDER, zero_division=0))
    cm = confusion_matrix(y_true, y_pred, labels=LABEL_ORDER)
    fig, ax = plt.subplots(figsize=(4.5, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
                xticklabels=LABEL_ORDER, yticklabels=LABEL_ORDER, ax=ax)
    ax.set_xlabel('Predicho'); ax.set_ylabel('Real'); ax.set_title(titulo)
    if fname: save_fig(fname)
    plt.show()
    return f1

# Acumulador de resultados en disco: cada sección (que puede correr en un kernel/entorno distinto)
# graba su F1 (val/test) y recall por clase (test). La celda de consolidación lee este JSON.
RESULTS_JSON = BASE_DIR / 'reports' / 'n1_results.json'
if not RESULTS_JSON.exists():
    RESULTS_JSON.write_text('{}')

def record_result(name, y_val_true, y_val_pred, y_test_true, y_test_pred):
    data = json.loads(RESULTS_JSON.read_text())
    cls_v = [c for c in LABEL_ORDER if c in set(y_val_true)]
    cls_t = [c for c in LABEL_ORDER if c in set(y_test_true)]
    rec = recall_score(y_test_true, y_test_pred, labels=LABEL_ORDER, average=None, zero_division=0)
    data[name] = {
        'f1_val':  round(float(f1_score(y_val_true,  y_val_pred,  labels=cls_v, average='macro')), 4),
        'f1_test': round(float(f1_score(y_test_true, y_test_pred, labels=cls_t, average='macro')), 4),
        'recall_test': {c: round(float(r), 4) for c, r in zip(LABEL_ORDER, rec)},
    }
    RESULTS_JSON.write_text(json.dumps(data, indent=2))
    print(f'[registrado] {name}: f1_val={data[name]["f1_val"]}  f1_test={data[name]["f1_test"]}')
    return data[name]

print('Setup OK. Resultados se acumulan en:', RESULTS_JSON)

## 1. Re-etiquetado N+1 y diagnóstico

El CSV ya tiene `label` = decisión_N. Ordenando por fecha, `shift(-1)` asigna a cada minuta la
decisión de la reunión **siguiente**. Se cae el último documento (no tiene N+1). Mantenemos el split
cronológico; el target pasa a ser `label_next`.

La **autocorrelación** `label_N == label_{N+1}` mide la fuerza de la persistencia: cuánto de la próxima
decisión se explica con sólo repetir la actual.

In [ ]:
df = pd.read_csv(DATA_PROCESSED / 'fomc_dataset.csv', parse_dates=['date']).sort_values('date').reset_index(drop=True)
df['label_next'] = df['label'].shift(-1)                        # decisión de la reunión N+1
df = df.dropna(subset=['label_next']).reset_index(drop=True)    # se cae el último doc (sin N+1)

train = df[df.split == 'train']
val   = df[df.split == 'val']
test  = df[df.split == 'test']

print('Documentos tras el shift:', len(df), '(esperado 208)')
print('train:', len(train), '| val:', len(val), '| test:', len(test), '(test pierde 1 -> 31)')
print()
print('Distribución del target N+1 por split:')
print(pd.crosstab(df['label_next'], df['split']))
print()
for nombre, d in [('global', df), ('train', train), ('val', val), ('test', test)]:
    frac = (d['label'].values == d['label_next'].values).mean()
    print(f'Persistencia (label_N == label_N+1) en {nombre:6s}: {frac:.1%}')

## 2. Baselines: mayoritaria y persistencia

- **Mayoritaria**: predice siempre la clase más frecuente de train (cota trivial).
- **Persistencia / momentum** (el rival real): para cada doc predice su decisión concurrente
  `label_N` como pronóstico de `label_{N+1}`. Cualquier modelo de texto útil debe **superarla**.

In [ ]:
# Baseline 1: clase mayoritaria (prior de train sobre el target N+1)
dummy = DummyClassifier(strategy='prior').fit(train['text'], train['label_next'])
pred_val_maj  = dummy.predict(val['text'])
pred_test_maj = dummy.predict(test['text'])
evaluar(val['label_next'],  pred_val_maj,  'Baseline mayoritaria (N+1) — VAL')
evaluar(test['label_next'], pred_test_maj, 'Baseline mayoritaria (N+1) — TEST', '09_baseline_confusion_test.png')
record_result('Baseline (mayoritaria)', val['label_next'], pred_val_maj, test['label_next'], pred_test_maj)

# Baseline 2 (LA VARA A VENCER): persistencia / momentum.
# Predicción = decisión concurrente (label_N); verdad = decisión siguiente (label_next).
evaluar(val['label_next'],  val['label'],  'Persistencia / momentum (N+1) — VAL')
evaluar(test['label_next'], test['label'], 'Persistencia / momentum (N+1) — TEST', '09_persistencia_confusion_test.png')
record_result('Persistencia (momentum)', val['label_next'], val['label'], test['label_next'], test['label'])

## 3. TF-IDF (léxico)

Mismo pipeline que el Exp 1 (notebook 04): `TfidfVectorizer(ngram_range=(1,2), max_features=10000,
sublinear_tf=True)` ajustado sólo en train, `LogisticRegression(class_weight='balanced')`, grid de `C`
sobre validación. Lo único que cambia respecto del Exp 1 es el target: `label_next`.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

vectorizer = TfidfVectorizer(ngram_range=(1, 2), max_features=10000, sublinear_tf=True)
X_train = vectorizer.fit_transform(train['text'])
X_val   = vectorizer.transform(val['text'])
X_test  = vectorizer.transform(test['text'])
y_train, y_val, y_test = train['label_next'], val['label_next'], test['label_next']

resultados = []
for C in [0.01, 0.1, 1, 10]:
    clf = LogisticRegression(C=C, max_iter=1000, class_weight='balanced').fit(X_train, y_train)
    f1v = f1_score(y_val, clf.predict(X_val),
                   labels=[c for c in LABEL_ORDER if c in set(y_val)], average='macro')
    resultados.append((C, f1v))
    print(f'C={C:<5} -> F1-macro val = {f1v:.4f}')
best_C = max(resultados, key=lambda t: t[1])[0]
print(f'\nMejor C (por F1-macro en val): {best_C}')

clf = LogisticRegression(C=best_C, max_iter=1000, class_weight='balanced').fit(X_train, y_train)
pred_val, pred_test = clf.predict(X_val), clf.predict(X_test)
evaluar(y_val,  pred_val,  f'TF-IDF + LogReg N+1 (C={best_C}) — VAL',  '09_tfidf_confusion_val.png')
evaluar(y_test, pred_test, f'TF-IDF + LogReg N+1 (C={best_C}) — TEST', '09_tfidf_confusion_test.png')
record_result('TF-IDF', y_val, pred_val, y_test, pred_test)

## 4. Word2Vec (semántica) — requiere venv py3.12 + gensim

Mismo pipeline que el Exp 3 (notebook 06): embeddings `word2vec-google-news-300`, mean pooling
ignorando OOV, `StandardScaler` + `LogisticRegression(class_weight='balanced')`, grid de `C`.

⚠️ Correr esta celda en el **venv py3.12** con los vectores GoogleNews ya cacheados (`~/gensim-data/`).
Antes ejecutá las celdas 0–1 (setup + relabel) en este mismo kernel.

In [ ]:
import gensim.downloader as api
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

wv = api.load('word2vec-google-news-300')
DIM = wv.vector_size
print('Word2Vec cargado. dim:', DIM)

def doc_vector(text, wv, dim):
    toks = [t for t in str(text).split() if t in wv]
    if not toks:
        return np.zeros(dim, dtype=np.float32)
    return np.mean([wv[t] for t in toks], axis=0)

X_train = np.vstack([doc_vector(t, wv, DIM) for t in train['text']])
X_val   = np.vstack([doc_vector(t, wv, DIM) for t in val['text']])
X_test  = np.vstack([doc_vector(t, wv, DIM) for t in test['text']])
y_train, y_val, y_test = train['label_next'], val['label_next'], test['label_next']

resultados = []
for C in [0.01, 0.1, 1, 10]:
    pipe = Pipeline([('scaler', StandardScaler()),
                     ('clf', LogisticRegression(C=C, max_iter=1000, class_weight='balanced'))]).fit(X_train, y_train)
    f1v = f1_score(y_val, pipe.predict(X_val),
                   labels=[c for c in LABEL_ORDER if c in set(y_val)], average='macro')
    resultados.append((C, f1v))
    print(f'C={C:<5} -> F1-macro val = {f1v:.4f}')
best_C = max(resultados, key=lambda t: t[1])[0]
print(f'\nMejor C (por F1-macro en val): {best_C}')

pipe = Pipeline([('scaler', StandardScaler()),
                 ('clf', LogisticRegression(C=best_C, max_iter=1000, class_weight='balanced'))]).fit(X_train, y_train)
pred_val, pred_test = pipe.predict(X_val), pipe.predict(X_test)
evaluar(y_val,  pred_val,  f'Word2Vec + LogReg N+1 (C={best_C}) — VAL',  '09_word2vec_confusion_val.png')
evaluar(y_test, pred_test, f'Word2Vec + LogReg N+1 (C={best_C}) — TEST', '09_word2vec_confusion_test.png')
record_result('Word2Vec', y_val, pred_val, y_test, pred_test)

## 5. FinBERT fine-tuning (contexto) — requiere venv CUDA

Mismo pipeline 4b que el notebook 07: `ProsusAI/finbert` + capa lineal, texto **crudo** de las
minutas, truncado head+tail (255+255), `WeightedTrainer` con class weights, early stopping por
F1-macro, grid `lr ∈ {1e-5,2e-5,5e-5} × batch ∈ {4,8,16}`. Target: `label_next`.

⚠️ Correr en el **venv CUDA (RTX 4070)**. Es autocontenida (recarga el CSV con texto crudo); sólo
necesita haber corrido la celda 0 (setup). ~1 min por config en GPU.

In [ ]:
import re
import torch
from torch import nn
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          TrainingArguments, Trainer, DataCollatorWithPadding,
                          EarlyStoppingCallback)

MODEL_NAME = 'ProsusAI/finbert'
device = torch.device('cuda' if torch.cuda.is_available() else
                      ('mps' if torch.backends.mps.is_available() else 'cpu'))
print('Device:', device)

# Datos: labels/splits + target N+1, con texto CRUDO de las minutas (autocontenido).
def raw_text(date):
    f = MINUTES_DIR / f"{pd.to_datetime(date).strftime('%Y-%m-%d')}_minutes.txt"
    return re.sub(r'\s+', ' ', f.read_text(encoding='utf-8', errors='replace')).strip()

dfb = pd.read_csv(DATA_PROCESSED / 'fomc_dataset.csv', parse_dates=['date']).sort_values('date').reset_index(drop=True)
dfb['label_next'] = dfb['label'].shift(-1)
dfb = dfb.dropna(subset=['label_next']).reset_index(drop=True)
dfb['raw'] = dfb['date'].apply(raw_text)

train_b = dfb[dfb.split == 'train']
val_b   = dfb[dfb.split == 'val']
test_b  = dfb[dfb.split == 'test']
y_val, y_test = val_b['label_next'], test_b['label_next']
print('train:', len(train_b), '| val:', len(val_b), '| test:', len(test_b))

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
CLS, SEP = tokenizer.cls_token_id, tokenizer.sep_token_id

def ids_head_tail(text, n_head=255, n_tail=255):
    ids = tokenizer(text, add_special_tokens=False, truncation=False)['input_ids']
    if len(ids) > n_head + n_tail:
        ids = ids[:n_head] + ids[-n_tail:]
    return [CLS] + ids + [SEP]

label2id = {l: i for i, l in enumerate(LABEL_ORDER)}
id2label = {i: l for l, i in label2id.items()}

class FinbertDataset(torch.utils.data.Dataset):
    def __init__(self, texts, labels):
        self.texts = list(texts); self.labels = [label2id[l] for l in labels]
    def __len__(self): return len(self.texts)
    def __getitem__(self, i):
        ids = ids_head_tail(self.texts[i])
        return {'input_ids': ids, 'attention_mask': [1]*len(ids), 'labels': self.labels[i]}

ds_train = FinbertDataset(train_b['raw'], train_b['label_next'])
ds_val   = FinbertDataset(val_b['raw'],   val_b['label_next'])
ds_test  = FinbertDataset(test_b['raw'],  test_b['label_next'])
collator = DataCollatorWithPadding(tokenizer)

counts = train_b['label_next'].value_counts()
w = torch.tensor([len(train_b) / (len(LABEL_ORDER) * counts[l]) for l in LABEL_ORDER], dtype=torch.float)
print('Class weights:', dict(zip(LABEL_ORDER, w.tolist())))

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop('labels')
        outputs = model(**inputs)
        loss = nn.CrossEntropyLoss(weight=w.to(model.device))(outputs.logits, labels)
        return (loss, outputs) if return_outputs else loss

def compute_metrics(p):
    preds = np.argmax(p.predictions, axis=-1)
    return {'f1_macro': f1_score(p.label_ids, preds, average='macro')}

In [ ]:
def train_one(lr, batch_size, epochs=5):
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=3, id2label=id2label, label2id=label2id).to(device)
    args = TrainingArguments(
        output_dir=str(BASE_DIR / 'tmp_finbert_n1' / f'lr{lr}_bs{batch_size}'),
        learning_rate=lr, per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=16, num_train_epochs=epochs,
        eval_strategy='epoch', save_strategy='epoch', logging_strategy='epoch',
        load_best_model_at_end=True, metric_for_best_model='f1_macro', greater_is_better=True,
        weight_decay=0.01, seed=33, report_to='none', disable_tqdm=True, save_total_limit=1)
    trainer = WeightedTrainer(
        model=model, args=args, train_dataset=ds_train, eval_dataset=ds_val,
        data_collator=collator, compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)])
    trainer.train()
    return trainer, trainer.evaluate(ds_val)['eval_f1_macro']

best_ft = None  # (f1, lr, batch, trainer)
for lr in [1e-5, 2e-5, 5e-5]:
    for bs in [4, 8, 16]:
        print(f'== lr={lr} batch={bs} ==')
        trainer, f1v = train_one(lr, bs)
        print(f'   F1-macro val = {f1v:.4f}')
        if best_ft is None or f1v > best_ft[0]:
            best_ft = (f1v, lr, bs, trainer)
print(f'\nMejor fine-tuning: lr={best_ft[1]} batch={best_ft[2]} -> F1-macro val {best_ft[0]:.4f}')

def predict_labels(trainer, ds):
    logits = trainer.predict(ds).predictions
    return [id2label[i] for i in np.argmax(logits, axis=-1)]

pred_val  = predict_labels(best_ft[3], ds_val)
pred_test = predict_labels(best_ft[3], ds_test)
evaluar(y_val,  pred_val,  f'FinBERT FT N+1 (lr={best_ft[1]}, batch={best_ft[2]}) — VAL',  '09_finbert_confusion_val.png')
evaluar(y_test, pred_test, f'FinBERT FT N+1 (lr={best_ft[1]}, batch={best_ft[2]}) — TEST', '09_finbert_confusion_test.png')
record_result('FinBERT FT', list(y_val), pred_val, list(y_test), pred_test)

## 6. Consolidación: N+1 vs N (concurrente)

Lee `reports/n1_results.json` (lo que se haya corrido en cualquier kernel) y arma la tabla y figuras.
Compara el F1-macro de test de la tarea **N+1** contra la **persistencia** (la vara) y contra la tarea
**concurrente N** (números conocidos de los Exp 1/3/4b).

In [ ]:
import json
data = json.loads(RESULTS_JSON.read_text())

orden = ['Baseline (mayoritaria)', 'Persistencia (momentum)', 'TF-IDF', 'Word2Vec', 'FinBERT FT']
# Números conocidos de la tarea CONCURRENTE (N) — F1-macro test (notebook 08)
concurrente_test = {'Baseline (mayoritaria)': 0.213, 'TF-IDF': 0.213, 'Word2Vec': 0.337, 'FinBERT FT': 0.648}

filas = [(m, data[m]['f1_val'], data[m]['f1_test'], concurrente_test.get(m, np.nan))
         for m in orden if m in data]
tab = pd.DataFrame(filas, columns=['modelo', 'f1_val (N+1)', 'f1_test (N+1)', 'f1_test (N concurrente)']).set_index('modelo')
display(tab.round(3))

faltan = [m for m in orden if m not in data]
if faltan:
    print('OJO — todavía faltan correr:', faltan)

present = [m for m in orden if m in data]

# Figura 1: F1-macro test N+1 vs N concurrente, con la vara de persistencia
x = np.arange(len(present)); width = 0.4
fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(x - width/2, [data[m]['f1_test'] for m in present], width, label='N+1 (test)', color='#fc8d62')
ax.bar(x + width/2, [concurrente_test.get(m, np.nan) for m in present], width, label='N concurrente (test)', color='#8da0cb')
if 'Persistencia (momentum)' in data:
    ax.axhline(data['Persistencia (momentum)']['f1_test'], ls='--', c='red', lw=1.3,
               label=f"Persistencia N+1 = {data['Persistencia (momentum)']['f1_test']:.2f}")
ax.set_xticks(x); ax.set_xticklabels([m.replace(' ', '\n') for m in present], fontsize=8)
ax.set_ylabel('F1-macro'); ax.set_ylim(0, 1)
ax.set_title('Predecir la PRÓXIMA decisión (N+1) vs la concurrente (N) — F1-macro test')
ax.legend()
plt.tight_layout(); save_fig('09_comparacion_n_vs_n1.png'); plt.show()

# Figura 2: recall por clase en test (tarea N+1)
recall_df = pd.DataFrame(
    [[data[m]['recall_test'][c] for c in LABEL_ORDER] for m in present],
    index=present, columns=LABEL_ORDER)
fig, ax = plt.subplots(figsize=(7, 4.2))
sns.heatmap(recall_df, annot=True, fmt='.2f', cmap='Greens', vmin=0, vmax=1,
            cbar_kws={'label': 'recall'}, ax=ax)
ax.set_title('Recall por clase en TEST — tarea N+1'); ax.set_xlabel('clase'); ax.set_ylabel('')
plt.tight_layout(); save_fig('09_recall_n1_test.png'); plt.show()

## 7. Interpretación (completar con los números obtenidos)

Plantilla de lectura una vez corrido todo:

- **¿Algún modelo supera la persistencia en F1-macro de test?**
  - **Sí** → la minuta contiene forward guidance real: el lenguaje anticipa el próximo movimiento más
    allá de la simple inercia. Hallazgo fuerte para la presentación.
  - **No** → las decisiones N+1 están dominadas por la persistencia del régimen; el texto aporta poco
    por encima de "saber qué hizo la Fed esta vez". Hallazgo honesto e igualmente presentable.
- **N+1 vs N (concurrente):** se espera que el F1-macro caiga al mirar a futuro (predecir es más difícil
  que reconstruir). Cuánto cae mide qué parte de la señal del Exp 1–4b era *explicativa* (describir la
  reunión actual) y qué parte es *anticipatoria*.
- **Recall por clase (test):** mirar especialmente `cut` y `hike` — si FinBERT FT mantiene recall de
  `cut` razonable al predecir N+1, es evidencia de que el forward guidance de recortes ya aparece en la
  minuta previa.

> Nota metodológica: la autocorrelación `label_N == label_{N+1}` (celda 1) es alta porque la Fed se
> mueve en ciclos; por eso la persistencia —y no la mayoritaria— es la vara correcta.